# Fase 2: Exploración, Limpieza Interanual, Transformación y Validación (`Base_Maestra_2023_2025.xlsx`)
**Curso:** MCDI500 — Programación para la Ciencia de Datos (Grupo 3)  
**Integrantes:** Madeleine Aguilera, Victoria Álvarez, Ignacio Peña y Raúl Hormazábal
 
---
## 1. Configuración del Entorno e Importación del Módulo Reutilizable (`F2/src/preprocesamiento.py`)


In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
 
# Asegurar que el directorio de trabajo sea la raíz del repositorio
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent.parent)
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))
 
from F2.src.preprocesamiento import (
    cargar_datos_seguro,
    diagnosticar_variables,
    estandarizar_regiones_y_peligrosidad,
    rescatar_coordenadas_chile,
    comparar_estrategias_nulos,
    imputar_por_grupo_con_flag,
)
 
sns.set_theme(style="whitegrid", palette="deep")
RUTA_RAW = Path("F1/data/raw/Base_Maestra_2023_2025.xlsx")
df_raw = cargar_datos_seguro(RUTA_RAW)
print(f"[OK] Dataset original cargado: {df_raw.shape[0]} filas x {df_raw.shape[1]} columnas")



## 2. Exploración Inicial, Diagnóstico de Calidad y Depuración Interanual (2023–2025)
Se inspecciona la estructura original del dataset (`118.280 filas × 21 columnas`), cuantificando:
1. **Registros duplicados exactos:** `118` filas duplicadas.
2. **Inconsistencias interanuales de formato:** `region` presenta `19` etiquetas para las 16 regiones de Chile y `peligrosidad` presenta `812` combinaciones por diferencias de mayúsculas, tildes y separadores (`##` en 2023–2024 vs `;` en 2025).
3. **Corrupción de escala decimal en coordenadas 2024:** En el año 2024 las coordenadas se exportaron sin separador decimal (ej. `-33447301481283` en lugar de `-33.447301`).


In [ ]:
# 2.1 Diagnóstico cuantitativo sobre el dataset crudo
tabla_diagnostico_raw = diagnosticar_variables(df_raw)
display(tabla_diagnostico_raw)
 
# 2.2 Eliminación de filas duplicadas exactas
n_duplicados = int(df_raw.duplicated().sum())
df_clean = df_raw.drop_duplicates().copy().reset_index(drop=True)
print(f"Duplicados exactos eliminados: {n_duplicados} -> Filas únicas: {len(df_clean)}")
 
# 2.3 Estandarización de regiones (19 -> 16) y peligrosidad (812 -> 93)
print(f"Antes de normalizar -> Regiones: {df_clean['region'].nunique()} | Peligrosidades: {df_clean['peligrosidad'].nunique()}")
df_clean = estandarizar_regiones_y_peligrosidad(df_clean)
print(f"Después de normalizar -> Regiones: {df_clean['region'].nunique()} | Peligrosidades: {df_clean['peligrosidad'].nunique()}")
 
# 2.4 Rescate de punto decimal y validación de límites geográficos de Chile en latitud y longitud
df_clean = rescatar_coordenadas_chile(df_clean)
tabla_diagnostico_pos = diagnosticar_variables(df_clean)
print("\nDiagnóstico de variables numéricas tras rescate de coordenadas:")
display(tabla_diagnostico_pos[tabla_diagnostico_pos["variable"].isin(["cantidad_kg", "latitud", "longitud"])])


## 3. Estrategia Fundamentada de Manejo de Datos Faltantes e Imputación con Trazabilidad
- **Descarte de `dropna()`:** Eliminar filas con valores faltantes descartaría **5.037 declaraciones (4,26% del total)**, omitiendo miles de toneladas de residuos peligrosos reportados al RETC.
- **Descarte de la media global en `cantidad_kg`:** La variable `cantidad_kg` presenta un **16,19% de valores atípicos extremos** (grandes complejos industriales y mineros con declaraciones de hasta 48.223 toneladas), lo que eleva la media a **16.422,33 kg** frente a una mediana de **234,00 kg** (70 veces menor). Imputar con la media inflaría artificialmente los registros faltantes.
- **Decisión adoptada:** Imputar las variables continuas (`cantidad_kg`, `latitud`, `longitud`) con la **mediana agrupada por `region`** y las categóricas (`id_lista_a`, `id_ciiu4`, `ciiu4`, `id_ciiu6`, `ciiu6`, `rubro`) con la **moda agrupada por `id_rubro`**, registrando columnas binarias `_imputado` (`0`/`1`).


In [ ]:
# 3.1 Tabla comparativa de estrategias de manejo de nulos sobre cantidad_kg y latitud
print("Comparación de estrategias sobre 'cantidad_kg':")
display(comparar_estrategias_nulos(df_clean, col_num="cantidad_kg", col_grupo="region"))
 
print("\nComparación de estrategias sobre 'latitud':")
display(comparar_estrategias_nulos(df_clean, col_num="latitud", col_grupo="region"))
 
# 3.2 Aplicar imputación por grupo con creación de columnas binarias '_imputado'
COLS_NUM_CON_NULOS = ["cantidad_kg", "latitud", "longitud"]
COLS_CAT_CON_NULOS = ["id_lista_a", "id_ciiu4", "ciiu4", "id_ciiu6", "ciiu6", "rubro"]
 
df_imputado = imputar_por_grupo_con_flag(
    df_clean,
    cols_numericas=COLS_NUM_CON_NULOS,
    cols_categoricas=COLS_CAT_CON_NULOS,
    col_grupo_num="region",
    col_grupo_cat="id_rubro",
)
 
print(f"\n[OK] Total de valores nulos tras la imputación: {int(df_imputado.isna().sum().sum())}")
 
# 3.3 Visualización: distribución log10(cantidad_kg) y coordenadas rescatadas e imputadas
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.histplot(np.log10(df_imputado["cantidad_kg"].clip(lower=1)), bins=40, kde=True, ax=axes[0], color="#1d4ed8")
axes[0].set_title("Distribución de log10(cantidad_kg) tras imputación")
axes[0].set_xlabel("log10(cantidad_kg)")
 
sns.boxplot(data=df_imputado, x="latitud", y="estado_materia", ax=axes[1], palette="Set2")
axes[1].set_title("Distribución geográfica (latitud) por estado físico del residuo")
plt.tight_layout()
plt.show()
